---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\Carmen Ciutu\Desktop\proiect AI\echochamber-project-team-4
DeepSeek key: True
Gemini key: True
Model folosit: deepseek-chat
OK


## Corpus

In [3]:
import pandas as pd
import random
from pathlib import Path

# Construim calea absolută către corpus, indiferent de directorul curent.
try:
    project_root = ROOT
except NameError:
    project_root = Path.cwd()
    while not (project_root / ".env").exists() and project_root.parent != project_root:
        project_root = project_root.parent

corpus_path = project_root / "data" / "cleaned" / "corpus_youtube_sample.jsonl"
if not corpus_path.exists():
    raise FileNotFoundError(f"Nu găsesc fișierul: {corpus_path}")

corpus = pd.read_json(corpus_path, lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[georgesimionoficial] Vai ce ruj are diva artista cu intalnirea lui Simion galeristul!
[turcescu111] Clarificati sursele de venit ale domnului . Răspunsul dat doamnei Mars cu Dumnez
[faiarsilviu] La 12 noaptea, pe o sosea din București, am văzut niste persoane care aruncau pe


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte

AXA_1 = "elite_blame"
AXA_2 = "national_identity"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
elite_blame măsoară dacă textul atribuie vina „elitelor” (politicieni, guvern,
partide, instituții centrale, „sistem”, „Bruxelles”, „oculta”) pentru probleme sociale/politice.
0 = absent (nu apare ideea de vină a elitelor)
1 = prezent (apare punctual acuzația)
2 = dominant (mesajul este centrat pe vina elitelor)
"""

AXA_2_DEFINITION = """
national_identity măsoară dacă textul folosește explicit identitatea națională
(„români”, „neam”, „țară”, „patrie”, „suveranitate”, tradiții etc) ca argument politic.
0 = absent (nu apare referință la identitate națională)
1 = prezent (apare ca element secundar)
2 = dominant (identitatea națională este cadrul principal al mesajului)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [8]:
MINI_PROMPT = f"""
Ești un specialist care analizeaza discursul politic de pe YouTube

SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}

CÂMPURI OBLIGATORII:
- target = ținta politică principală din comentariu
- stance = poziția față de target: pro / anti / neutru / ambiguu / none
- tone = tonul dominant: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
- {AXA_1} = 0 / 1 / 2
- {AXA_2} = 0 / 1 / 2

DEFINIȚII AXE:
{AXA_1_DEFINITION}

{AXA_2_DEFINITION}

REGULI DE CODARE:
1. Codează doar ce apare explicit sau este inferabil direct din comentariu, titlu și canal.
2. Nu folosi informații externe și nu presupune context lipsă.
3. Dacă nu există o țintă politică clară: target='none' și stance='none'.
4. Dacă textul e ironic, codează sensul intenționat al mesajului.
5. Pentru axe folosește strict scala: 0=absent, 1=prezent, 2=dominant.
6. Returnează doar un JSON valid, fără explicații suplimentare, fără markdown.
7. Toate cheile trebuie să existe în output.

FORMAT OUTPUT (STRICT):
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""

print(MINI_PROMPT)


Ești un specialist care analizeaza discursul politic de pe YouTube

SARCINĂ:
Adnotează comentariul folosind două axe:
1. elite_blame
2. national_identity

CÂMPURI OBLIGATORII:
- target = ținta politică principală din comentariu
- stance = poziția față de target: pro / anti / neutru / ambiguu / none
- tone = tonul dominant: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
- elite_blame = 0 / 1 / 2
- national_identity = 0 / 1 / 2

DEFINIȚII AXE:

elite_blame măsoară dacă textul atribuie vina „elitelor” (politicieni, guvern,
partide, instituții centrale, „sistem”, „Bruxelles”, „oculta”) pentru probleme sociale/politice.
0 = absent (nu apare ideea de vină a elitelor)
1 = prezent (apare punctual acuzația)
2 = dominant (mesajul este centrat pe vina elitelor)



national_identity măsoară dacă textul folosește explicit identitatea națională
(„români”, „neam”, „țară”, „patrie”, „suveranitate”, tradiții etc) ca argument politic.
0 = absent (nu apare referință la identitate național

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [11]:
TESTS = corpus.sample(10, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [12]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [13]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [14]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "elite_blame": 1,
  "national_identity": 0
}
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
{
  "target": "Iran",
  "stance": "neutru",
  "tone": "acuzator",
  "elite_blame": 0,
  "national_identity": 0
}
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "elite_blame": 0,
  "national_identity": 1
}
COMENTARIU:
Nicușor merge pe lângă covor pentru că nu are ,, 

In [ ]:
## Pasul 6 — Interpretare scurtă

### 1. Ce două axe ai ales?
elite_blame și national_identity

### 2. De ce le-ai ales?
Am ales aceste axe pentru că sunt complementare și capturează forțele principale ale discursului populist-naționalist:
- elite_blame măsoară retorica anti-establishment (acuzații către guvern, sistem, instituții)
- national_identity măsoară Framing-ul patriotic/suveranist

Împreună, acestea ajută la identificarea și cuantificarea mesajelor populiste.

### 3. Modelul a returnat JSON corect?

- Promptul este suficient de clar și detaliat pentru a constrânge output-ul modelului
- JSON-urile returnate conțin cheile așteptate: target, stance, tone, elite_blame, national_identity


### 4. Care a fost cea mai mare problemă?
-
### 5. Ce ai schimba în prompt?
Nimic 